# Verify DSCI Annual Averages

Downloads weekly Drought Severity and Coverage Index (DSCI) readings from the
U.S. Drought Monitor API, computes calendar-year averages, and compares them to
values in `../data/wildfire-data.csv`.

**National:** `USStatistics/GetDSCI?aoi=conus` (2000 onward; 1999 has no API data)

**Western:** `NWSRegionStatistics/GetDSCI?aoi=WR` (NWS Western Region)

**Method:** Average all weekly DSCI values whose map date falls in each calendar
year. Partial 2026 uses readings through the latest available week.

**Output:** Updates `../data/dsci-annual-averages.csv`, `../data/dsci-western-annual.csv`,
and saves raw weekly downloads to `../data/dsci-source/`.

In [ ]:
import csv
import io
import urllib.request
from pathlib import Path

import pandas as pd

In [ ]:
FULL_YEARS = list(range(2000, 2026))
PARTIAL_YEAR = 2026
PARTIAL_END = "7/16/2026"
CONUS_API = "https://usdmdataservices.unl.edu/api/USStatistics/GetDSCI?aoi=conus&startdate=1/1/{year}&enddate=12/31/{year}"
WEST_API = "https://usdmdataservices.unl.edu/api/NWSRegionStatistics/GetDSCI?aoi=WR&startdate=1/1/{year}&enddate=12/31/{year}"
PARTIAL_CONUS = f"https://usdmdataservices.unl.edu/api/USStatistics/GetDSCI?aoi=conus&startdate=1/1/{PARTIAL_YEAR}&enddate={PARTIAL_END}"
PARTIAL_WEST = f"https://usdmdataservices.unl.edu/api/NWSRegionStatistics/GetDSCI?aoi=WR&startdate=1/1/{PARTIAL_YEAR}&enddate={PARTIAL_END}"
SOURCE_DIR = Path("../data/dsci-source")
AUDIT_PATH = Path("../data/dsci-annual-averages.csv")
WEST_PATH = Path("../data/dsci-western-annual.csv")
CHART_PATH = Path("../data/wildfire-data.csv")

In [ ]:
def download_csv(url):
    with urllib.request.urlopen(url, timeout=60) as resp:
        return resp.read().decode()


def annual_avg_from_csv(text, year):
    rows = list(csv.DictReader(io.StringIO(text)))
    vals = [int(r["DSCI"]) for r in rows if r["MapDate"].startswith(str(year))]
    if not vals:
        return None, 0
    return round(sum(vals) / len(vals), 1), len(vals)

In [ ]:
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
records = []

for year in FULL_YEARS:
    url = API.format(year=year)
    try:
        text = download_csv(url)
        out = SOURCE_DIR / f"dsci-{year}.csv"
        out.write_text(text)
        avg, weeks = annual_avg_from_csv(text, year)
        records.append({
            "year": year,
            "weeks": weeks,
            "dsci_avg": avg,
            "partial": False,
            "source": "USDM API USStatistics/GetDSCI aoi=conus",
        })
        print(f"{year}: {avg} ({weeks} weeks)")
    except Exception as e:
        print(f"{year}: FAILED: {e}")
        records.append({"year": year, "weeks": 0, "dsci_avg": None, "partial": False, "source": "FAILED"})

try:
    text = download_csv(PARTIAL_API)
    out = SOURCE_DIR / "dsci-2026-partial.csv"
    out.write_text(text)
    avg, weeks = annual_avg_from_csv(text, PARTIAL_YEAR)
    records.append({
        "year": PARTIAL_YEAR,
        "weeks": weeks,
        "dsci_avg": avg,
        "partial": True,
        "source": f"USDM API partial through {PARTIAL_END}",
    })
    print(f"{PARTIAL_YEAR} partial: {avg} ({weeks} weeks)")
except Exception as e:
    print(f"{PARTIAL_YEAR} partial: FAILED: {e}")

In [ ]:
audit = pd.DataFrame(records)
audit.to_csv(AUDIT_PATH, index=False)
print(f"Saved audit to {AUDIT_PATH}")

chart = pd.read_csv(CHART_PATH, skiprows=[1])
chart["year"] = chart["year"].astype(int)
chart_vals = chart[["year", "dsci_avg"]].dropna(subset=["dsci_avg"])
chart_vals["dsci_avg"] = chart_vals["dsci_avg"].astype(float)

merged = audit.merge(chart_vals, on="year", how="inner", suffixes=("_api", "_chart"))
merged["delta"] = merged["dsci_avg_api"] - merged["dsci_avg_chart"]
merged["match"] = merged["delta"].abs() < 0.15
print(merged[["year", "dsci_avg_api", "dsci_avg_chart", "delta", "match"]].to_string(index=False))

mismatches = merged[~merged["match"]]
if len(mismatches):
    print(f"\nWARNING: {len(mismatches)} year(s) differ from wildfire-data.csv")
else:
    print("\nAll chart DSCI values match API audit.")

## Source Citation

U.S. Drought Monitor, National Drought Mitigation Center, University of Nebraska-Lincoln.
DSCI via API: https://usdmdataservices.unl.edu/api/USStatistics/GetDSCI?aoi=conus

Area: Contiguous U.S., national level
Method: Calendar-year average of weekly DSCI readings (0-500 scale)